# Flood summaries for ThinkHazard

This script performs flood hazard ranking by administrative unit using global-extent
Fathom tiles hosted on AWS S3, rather than country-extent locally downloaded data.

The hazard ranking is based on:
- Value threshold: Minimum flood depth (cm) to consider
- Area threshold: Minimum percentage of area affected
- Hazard score: Count of return periods meeting both thresholds (0-3)


In [ ]:
import os, time, io, json, sys
import urllib3
import boto3

import geopandas as gpd
import pandas as pd
import numpy as np

from functools import reduce
from urllib3.exceptions import InsecureRequestWarning
from botocore import UNSIGNED
from botocore.config import Config
from tqdm.notebook import tqdm

# Import helper functions
from gfdrr_helper import *

urllib3.disable_warnings(InsecureRequestWarning)

def tPrint(s):
    """prints the time along with the message"""
    print("%s\t%s" % (time.strftime("%H:%M:%S"), s))

s3_client = boto3.client('s3', verify=False, config=Config(signature_version=UNSIGNED))

%load_ext autoreload
%autoreload 2

In [ ]:
local_folder = "C:/WBG/Work/Projects/ThinkHazard"
out_folder = os.path.join(local_folder, "FATHOM_summaries")
map_folder = os.path.join(local_folder, "FATHOM_maps")
for tF in [out_folder, map_folder]:
    if not os.path.exists(tF):
        os.makedirs(tF)
vrt_folder = r"C:\WBG\Work\data\FATHOM"
s3_bucket = "wbg-geography01"
s3_prefix = "FATHOM"
return_periods = [10, 100, 500, 1000]
flood_files = [
    ["FU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-FLUVIAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ["CU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ['PD', "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-PLUVIAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"]
]

admin_boundaries_file = r"C:\WBG\Work\data\ADMIN\NEW_WB_BOUNDS\FOR_PUBLICATION\crs_4326\parquet\WB_GAD_ADM2.parquet"
inA = gpd.read_parquet(admin_boundaries_file)

In [ ]:
cur_out_folder = os.path.join(out_folder, "FATHOM_Detailed")
if not os.path.exists(cur_out_folder):
    os.makedirs(cur_out_folder)

cur_map_folder = os.path.join(map_folder, "FATHOM_Detailed")
if not os.path.exists(cur_map_folder):
    os.makedirs(cur_map_folder)

with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    for sel_country in inA['ISO_A3'].unique(): #
        if sel_country:
            all_res = []
            out_file = os.path.join(cur_out_folder, f"FATHOM_ThinkHazard_summary_{sel_country}.csv")
            sel_a = inA[inA['ISO_A3'] == sel_country]                           
            if not os.path.exists(out_file) and not (sel_country in []): #"FJI",'RUS'
                tPrint(f"Processing country: {sel_country}")
                for lbl, raster_file in flood_files:
                    for return_period in return_periods:
                        tPrint(f"Processing {lbl} for {return_period} year return period")
                        sel_raster_file = raster_file.format(rp=return_period)
                        sel_raster = f"s3://{s3_bucket}/{s3_prefix}/{sel_raster_file}"
                        res_a = calculate_think_hazard_score(sel_a, sel_raster, 
                                                            depth_threshold=50, idx_col='ADM2CD_c',
                                                            all_touched=True, no_data=-32767.0)
                        res_a.rename(columns={'frac_area_flooded': f'frac_area_flooded_{lbl}_{return_period}yr',
                                            'mean_val': f'mean_val_{lbl}_{return_period}yr'
                                            }, inplace=True)
                        all_res.append(res_a)                    
                all_res_df = reduce(lambda left, right: pd.merge(left, right, on='ADM2CD_c', how='outer'), all_res) 
                all_res_df.to_csv(out_file, index=False)

                sel_a = inA[inA['ISO_A3'] == sel_country]                                   
                map_adm = pd.merge(sel_a, all_res_df, on='ADM2CD_c', how='left')
                #map_flood(map_adm, return_period=100, out_file=os.path.join(cur_map_folder, f"flood_map_{sel_country}_100yr.png"))
            else:
                tPrint(f"File already exists for {sel_country}, skipping...")

In [ ]:
# Process for all the urban extents
urban_extents_file = os.path.join(local_folder, "TH_scores_urban_2025.gpkg")
in_urban = gpd.read_file(urban_extents_file)
all_res = []
for lbl, raster_file in flood_files:
    for return_period in return_periods:
        tPrint(f"Processing {lbl} for {return_period} year return period")
        sel_raster_file = raster_file.format(rp=return_period)
        sel_raster = f"s3://{s3_bucket}/{s3_prefix}/{sel_raster_file}"
        urban_res = calculate_think_hazard_score(in_urban, sel_raster, 
                                        depth_threshold=50, idx_col='ID_UC_G0',
                                        all_touched=True, no_data=-32767.0)
        urban_res.rename(columns={'frac_area_flooded': f'frac_area_flooded_{lbl}_{return_period}yr',
                                            'mean_val': f'mean_val_{lbl}_{return_period}yr'
                                            }, inplace=True)
        all_res.append(urban_res)
all_res_df = reduce(lambda left, right: pd.merge(left, right, on='ID_UC_G0', how='outer'), all_res) 
all_res_df.to_csv(urban_extents_file.replace(".gpkg", "_results.csv"), index=False)

## What's up with Russia and Fiji?
There were errors when processing Russia and Fiji, need to gigure out why

In [ ]:
lbl, raster_template = flood_files[0]
sel_raster_file = raster_template.format(rp=100)
sel_raster = f"s3://{s3_bucket}/{s3_prefix}/{sel_raster_file}"
with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    sel_r = rasterio.open(sel_raster)

iso3 = 'FJI'
selA = inA[inA['ISO_A3'] == iso3]

In [ ]:
urban_res = calculate_think_hazard_score(selA, sel_raster, 
                                        depth_threshold=50, idx_col='ADM2CD_c',
                                        all_touched=True, no_data=-32767.0)

In [ ]:
urban_res

# Comparing results

In [ ]:
# Comparing the results with the FATHOM summary data
fathom_file = r"c:\WBG\Work\Projects\ThinkHazard\FATHOM_summaries\FATHOM_Detailed\FATHOM_ThinkHazard_summary_TUN.xlsx"
# Read the second sheet of the Excel file
fathom_df = pd.read_excel(fathom_file, sheet_name=1)
fathom_df = fathom_df.loc[:, ['ADM2CD_c', 'CU_RP1000_mean', 'CU_RP1000_affected_pct']]

# Read in new results for Tunisia
new_results_file = r"c:\WBG\Work\Projects\ThinkHazard\FATHOM_summaries\FATHOM_Detailed\FATHOM_ThinkHazard_summary_TUN.csv"
new_results_df = pd.read_csv(new_results_file)
new_results_df = new_results_df.loc[:, ['ADM2CD_c', 'mean_val_CU_1000yr', 'frac_area_flooded_CU_1000yr']]   

# Merge the two dataframes on the ADM2CD_c column
merged_df = pd.merge(fathom_df, new_results_df, on='ADM2CD_c', how='inner')
# Calculate the differences between the two sets of results
merged_df['mean_diff'] = (merged_df['CU_RP1000_mean'] - merged_df['mean_val_CU_1000yr']) / merged_df['CU_RP1000_mean'] * 100
merged_df['frac_diff'] = (merged_df['CU_RP1000_affected_pct'] - merged_df['frac_area_flooded_CU_1000yr']) / merged_df['CU_RP1000_affected_pct'] * 100

merged_df.sort_values('mean_diff', inplace=True)

In [ ]:
merged_df.sort_values('frac_diff', ascending=False).head(10)

In [ ]:
merged_df.loc[~merged_df['frac_diff'].isna()].sort_values('frac_diff', ascending=False).tail(5)

# DEBUGGING

In [ ]:
from rasterio.windows import from_bounds
from rasterio.features import rasterize
from affine import Affine

In [ ]:
sel_country = 'TUN'
raster_path = "s3://wbg-geography01/FATHOM/FLOOD_MAP-1ARCSEC-NW_OFFSET-1in100-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"
inD = inA[inA['ISO_A3'] == sel_country].copy()
depth_threshold=50
idx_col='ADM2CD_c'
all_touched=True
min_val=None
max_val=None
no_data=[-32767.,-32768.]

with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
    curRaster = rasterio.open(raster_path)
    fCount = 0
    res = {}    
    #for idx, row in inD.iterrows():
    for unq_id in ['TUN004012', 'TUN024007', 'TUN022014', 'TUN016001', 'TUN013001', 'TUN015004', 'TUN016015']:
        out_folder = rf"C:\Temp\FATHOM_TUN\{unq_id}"
        if not os.path.exists(out_folder):
            os.makedirs(out_folder)

        row = inD.loc[inD['ADM2CD_c'] == unq_id].iloc[0]
        geometry = row["geometry"]
        fCount = fCount + 1
    
        window = from_bounds(*geometry.bounds, transform=curRaster.transform)
        shifted_affine = curRaster.window_transform(window)
        raw_data = curRaster.read(1, window=window)        
        data = np.where(np.isin(raw_data, no_data), np.nan, raw_data)        
                
        mask = rasterize(
            [(geometry, 0)],
            out_shape=data.shape,
            transform=shifted_affine,
            all_touched=all_touched,
            fill=1,
            dtype=np.uint8
        )
        # Add to the mask areas that are nan in the data
        mask = np.where(np.isnan(data), 1, mask)
        
        # create a masked numpy array
        masked_data = np.ma.array(data=data, mask=mask.astype(bool))
        # calculate area percentage above threshold
        area_flooded = (masked_data > depth_threshold).sum()
        
        #Write clip to file
        with rasterio.open(os.path.join(out_folder, f"{unq_id}_raw_data.tif"), 'w', driver='GTiff',
                height=raw_data.shape[0], width=raw_data.shape[1],
                count=1, dtype=raw_data.dtype,
                crs=curRaster.crs, transform=shifted_affine) as dst:
            dst.write(raw_data, 1)
        
        with rasterio.open(os.path.join(out_folder, f"{unq_id}_mask.tif"), 'w', driver='GTiff',
                        height=masked_data.shape[0], width=masked_data.shape[1],
                        count=1, dtype=masked_data.dtype,
                        crs=curRaster.crs, transform=shifted_affine) as dst:
            dst.write(masked_data.mask, 1)

        with rasterio.open(os.path.join(out_folder, f"{unq_id}_masked_data.tif"), 'w', driver='GTiff',
                        height=masked_data.shape[0], width=masked_data.shape[1],
                        count=1, dtype=masked_data.dtype,
                        crs=curRaster.crs, transform=shifted_affine) as dst:
            dst.write(masked_data.filled(nodata_value), 1)
    
    

        
    


In [ ]:
masked_data

In [ ]:
(masked_data > depth_threshold).sum()

In [ ]:
masked_data.shape[0] * masked_data.shape[1]

In [ ]:
masked_data.mask.sum()

# Extracting sample data

In [ ]:
sys.path.insert(0, "C:\\WBG\\Work\\Code\\GOSTrocks\\src")
import GOSTrocks.rasterMisc as rMisc

In [ ]:
temp_out_folder = "C:/Temp/FATHOM_TUN"
if not os.path.exists(temp_out_folder):
    os.makedirs(temp_out_folder)

sel_admin = inA.loc[inA['ISO_A3'] == "TUN"]
sel_admin.to_file(os.path.join(temp_out_folder, "TUN_admin.gpkg"), driver="GPKG")

return_periods = [10, 100, 500, 1000]
flood_files = [
    ["FU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-FLUVIAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ["CU", "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-COASTAL-UNDEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"],
    ['PD', "FLOOD_MAP-1ARCSEC-NW_OFFSET-1in{rp}-PLUVIAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.1.vrt"]
]

for return_period in return_periods:
    for lbl, raster_file in flood_files:
        temp_out_file = os.path.join(temp_out_folder, f"TUN_{lbl}_{return_period}yr.tif")
        if not os.path.exists(temp_out_file):
            sel_raster_file = raster_file.format(rp=return_period)
            sel_raster = f"s3://{s3_bucket}/{s3_prefix}/{sel_raster_file}"
            with rasterio.Env(GDAL_HTTP_UNSAFESSL='YES'):
                inR = rasterio.open(sel_raster)
                rMisc.clipRaster(inR, sel_admin, temp_out_file, crop=False)